# tsbench walkthrough: can you trust a forecasting leaderboard?

*By Krishna ([@Krishna-cpp](https://github.com/Krishna-cpp)).*

This notebook walks through the core ideas of the benchmark in about 5 minutes, using the
same code as the full run. No GPU needed.

1. Look at the three datasets and why they were chosen
2. See why a tree model fails on a trend, and the one-line fix
3. Run a rolling-origin backtest with conformal prediction intervals
4. Ask whether the winner is *significantly* better

In [ ]:
# On Colab: install the package straight from GitHub (takes ~1 minute)
import sys

if "google.colab" in sys.modules:
    !pip install -q git+https://github.com/Krishna-cpp/ts-benchmark.git

In [ ]:
import matplotlib.pyplot as plt

from tsbench.data import load_datasets

datasets = load_datasets()
fig, axes = plt.subplots(3, 1, figsize=(10, 7))
for ax, ds in zip(axes, datasets):
    ds.series.plot(ax=ax, lw=1)
    ax.set_title(f"{ds.name}: {ds.description}", loc="left", fontsize=10)
fig.tight_layout()

## 1. Trees cannot extrapolate

A gradient-boosted tree predicts averages of values it saw in training. CO2 keeps rising, so the
next value is *always* higher than anything in the training data, which a tree cannot produce.
Training the same model on the month-to-month **change** removes the trend and fixes it.

In [ ]:
from tsbench.metrics import mase
from tsbench.models import GBMLags

co2 = datasets[0]
y = co2.values
origin = len(y) - 12
train, actual = y[:origin], y[origin:]

level = GBMLags(24, difference=False).forecast(train, 12)
diff = GBMLags(24, difference=True).forecast(train, 12)

plt.figure(figsize=(10, 4))
plt.plot(range(-48, 0), train[-48:], color="gray", label="history")
plt.plot(range(12), actual, color="black", lw=2.5, label="actual")
plt.plot(range(12), level, label=f"GBM on levels  MASE={mase(actual, level, train, 12):.2f}")
plt.plot(range(12), diff, label=f"GBM on changes MASE={mase(actual, diff, train, 12):.2f}")
plt.axhline(train.max(), ls=":", color="red", label="max of training data")
plt.legend()
plt.title("Same model, one-line difference")

## 2. Rolling-origin backtest

One train/test split is one roll of the dice. Instead the forecast origin walks forward through
time and every model is refit at each step on the past only. Each forecast also gets an 80%
interval, calibrated by backtesting *inside* the training data (split conformal prediction).

In [ ]:
from tsbench.backtest import rolling_origin_backtest
from tsbench.intervals import IntervalConfig
from tsbench.models import build_models
from tsbench.stats import summarize

models = build_models(co2, ["Naive", "SeasonalNaive", "ETS", "SARIMA", "GBM-level", "GBM-diff"])
res = rolling_origin_backtest(co2.with_folds(5), models,
                              intervals=IntervalConfig(n_windows=5), n_jobs=-1)
res.folds[["model", "fold", "mase", "coverage"]].head(12)

## 3. Is the winner really better?

All models are scored on the *same* folds, so we can use a paired test (Wilcoxon signed-rank)
against the best model, with a Holm correction for testing several models at once.

In [ ]:
summary = summarize(res.folds)
summary[["model", "mase", "ci_low", "ci_high", "mean_rank", "p_vs_best",
         "significantly_worse", "coverage"]].round(3)

With only 5 folds the test has little power, so several models come out "tied" with the best.
That is the honest answer, and the reason the full benchmark uses 10 to 12 folds.

**Next:** run the full benchmark with `python -m tsbench --with-ttm` to add IBM's Granite
TinyTimeMixer (zero-shot and fine-tuned), or explore the results in the interactive explorer.